# Pelajaran 05 - Agentic RAG


## Setup

Notebook ini menunjukkan corak Agentic RAG (Retrieval-Augmented Generation) menggunakan Microsoft Agent Framework.

**Prasyarat:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — titik hujung perkhidmatan Azure AI Search anda
- `AZURE_SEARCH_API_KEY` — kunci API Azure AI Search anda
- Penempatan Azure OpenAI dikonfigurasi melalui pembolehubah persekitaran
- Azure CLI diautentikasi (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Apa itu Agentic RAG?

RAG tradisional mengikuti saluran tetap: ambil dokumen, kemudian jana jawapan. **Agentic RAG** melangkah lebih jauh dengan memberi agen autonomi untuk menentukan **bila** dan **bagaimana** untuk mengambil maklumat.

Dengan Agentic RAG, agen boleh:
- **Memutuskan** sama ada pengambilan diperlukan sebelum menjawab soalan
- **Memilih** sumber data atau alat yang hendak dirujuk
- **Menilai** hasil yang diperoleh dan melakukan pengambilan susulan jika cubaan pertama tidak mencukupi
- **Menggabungkan** maklumat dari beberapa langkah pengambilan ke dalam jawapan yang koheren

Ini menjadikan agen lebih fleksibel dan tepat berbanding saluran ambil kemudian jana yang statik.


## Mencipta Alat Carian

Dalam Agentic RAG, sumber data luaran dibalut sebagai **alat** yang boleh dipanggil oleh agen mengikut permintaan. Ini membolehkan agen menganggap pengambilan sebagai satu lagi tindakan yang boleh diambil, bukannya satu langkah wajib.

Di bawah kami mentakrifkan pangkalan pengetahuan perjalanan dan memaparkannya sebagai alat yang boleh dipanggil oleh agen untuk mendapatkan maklumat destinasi.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Membina Ejen RAG

Sekarang kita membina ejen yang diarahkan untuk **sentiasa mendapatkan maklumat sebelum menjawab**. Ejen tersebut menggunakan alat `search_travel_knowledge` untuk menguatkan jawapannya berdasarkan pangkalan pengetahuan dan bukannya bergantung pada data latihan sendiri.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Pengambilan Iteratif — Corak Pembuat-Pemeriksa

Satu kelebihan utama Agentic RAG ialah **pengambilan iteratif**. Ejen boleh melakukan beberapa pusingan carian untuk mengesahkan, memperhalusi, atau mengembangkan penemuan awalnya — serupa dengan aliran kerja "pembuat-pemeriksa":

1. **Langkah pembuat**: Ejen mengambil maklumat awal dan draf jawapan.
2. **Langkah pemeriksa**: Ejen melakukan pengambilan tambahan untuk mengesahkan butiran atau mengisi kekosongan.

Di bawah, ejen ditanya satu soalan yang memerlukan perbandingan beberapa destinasi, yang mendorongnya untuk mencari beberapa kali.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Summary

Dalam pelajaran ini anda belajar bagaimana untuk membina sistem **Agentic RAG** menggunakan Microsoft Agent Framework:

- **Agentic RAG** membolehkan ejen membuat keputusan secara autonomi bila untuk mendapatkan maklumat, menjadikan pengambilan maklumat dinamik dan bukannya tetap.
- **Alat sebagai sumber data**: Pangkalan data luaran (seperti Azure AI Search) dibungkus sebagai alat yang boleh dipanggil oleh ejen.
- **Pengambilan secara iteratif**: Corak maker-checker membolehkan ejen melakukan beberapa pusingan pengambilan — mencari, mengesah, dan memperhalusi — sebelum menghasilkan jawapan akhir.

Dalam pengeluaran, anda akan menggantikan `TRAVEL_KNOWLEDGE_BASE` dalam memori dengan indeks Azure AI Search sebenar untuk mengendalikan pengambilan dokumen perjalanan berskala besar.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan profesional oleh manusia adalah disyorkan. Kami tidak bertanggungjawab atas sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
